<a href="https://github.com/N3iKos/segsmaker-fast">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a>

---
> **Segsmaker — Optimized Fork (Colab)** · Philosophy: *Speed is a feature. Efficiency is the standard.*
>
> Run cells **top-to-bottom** on first use. On subsequent sessions, skip directly to **Launch**.

In [ ]:
# @title 🖥️ **WebUI Installer** {"display-mode":"form"}
# @markdown ### Step 1 — Pick your WebUI
Webui = 'Forge' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown ---
# @markdown ### Step 2 — API Keys
# @markdown > 🔑 Get Civitai key → https://civitai.com/user/account
Civitai_Key = '' # @param {type:"string", placeholder:"Your Civitai API Key (required)"}
# @markdown > 🤗 Get HF token → https://huggingface.co/settings/tokens
HF_Read_Token = '' # @param {type:"string", placeholder:"Your Huggingface READ Token (optional)"}
# @markdown ---
# @markdown ### Step 3 — Google Drive *(optional — for persistent storage)*
# @markdown > If **Yes**, models are saved to `MyDrive/Segsmaker/` and survive session resets.
Mount_GDrive = 'No' # @param ["Yes", "No"]

import subprocess, sys, os
from pathlib import Path

if Mount_GDrive == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

# Download setup.py from our fork and run it
_setup_py = '/content/setup.py'
_url = 'https://raw.githubusercontent.com/N3iKos/segsmaker-fast/main/script/KC/setup.py'
_r = subprocess.run(['curl', '-fLo', _setup_py, _url], capture_output=True, text=True)
if _r.returncode != 0:
    print(f'❌ Setup script download failed:\n{_r.stderr}')
    sys.exit(1)

print('✅ Setup script downloaded. Running installer...')
get_ipython().run_line_magic('run', f'{_setup_py} --webui="{Webui}" --civitai_key="{Civitai_Key}" --hf_read_token="{HF_Read_Token}"')

# Google Drive symlinks (only if mounted)
if Mount_GDrive == 'Yes':
    d = Path('/content/drive/MyDrive/Segsmaker')
    for n, p in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        f = d / n
        f.mkdir(parents=True, exist_ok=True)
        s = p / f'drive-{n}'
        if not s.exists(): s.symlink_to(f, target_is_directory=True)

    !rm -rf $WebUI_Output
    o = d / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    o.mkdir(parents=True, exist_ok=True)
    WebUI_Output.symlink_to(o, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        wc = WebUI / 'cache'
        !rm -rf $wc
        c = d / 'cache'
        c.mkdir(parents=True, exist_ok=True)
        wc.symlink_to(c, target_is_directory=True)

## 📥 Model Downloader
<span style="font-size:13px;">
Fill in the slots below. <b>Empty slots are automatically skipped.</b><br>
Supported: <code>civitai.com</code>, <code>huggingface.co</code>, direct URLs, Google Drive.<br>
Models saved to your persistent storage folder.
</span>

In [ ]:
# @title 📥 **Model Downloader** — 5 Checkpoint + 5 LoRA + 1 VAE {"display-mode":"form"}
# @markdown ### 🗃️ Checkpoints
Checkpoint_1 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_2 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_3 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_4 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_5 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### 🎨 LoRA
Lora_1 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_2 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_3 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_4 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_5 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### 🎛️ VAE
VAE_URL = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### ⚡ Speed Options
# @markdown > **Parallel Mode** = download all files simultaneously. Recommended: ON.
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:6, step:1}

from nenen88 import parallel_batch_download

_queue = []
for _url in [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]:
    if _url.strip(): _queue.append((_url.strip(), str(CKPT), None))
for _url in [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]:
    if _url.strip(): _queue.append((_url.strip(), str(LORA), None))
if VAE_URL.strip(): _queue.append((VAE_URL.strip(), str(VAE), None))

if not _queue:
    print('  No URLs provided — skipping.')
elif Parallel_Download:
    parallel_batch_download(_queue, max_workers=Max_Workers)
else:
    for _url, _dest, _fn in _queue:
        %cd -q $_dest
        %download $_url

## 🛠️ Extra Assets *(Optional)*
<span style="font-size:13px;">Extensions / Custom Nodes, Embeddings, Upscalers.</span>

In [ ]:
# @title 🛠️ **Extra Assets** {"display-mode":"form"}
# @markdown ### 🔌 Extensions / Custom Nodes (git clone URL)
Extension_1 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_2 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_3 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
# @markdown ---
# @markdown ### 🖼️ Embeddings
Embedding_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### 🔬 Upscalers
Upscaler_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### ⚡ Speed Options
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:6, step:1}

import tempfile, os
from nenen88 import parallel_batch_download

# Extensions — write to temp file, use %clone magic
_ext_urls = [u.strip() for u in [Extension_1, Extension_2, Extension_3] if u.strip()]
if _ext_urls:
    _tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False)
    _tmp.write('\n'.join(_ext_urls))
    _tmp.flush(); _tmp.close()
    print('\n⚡ Cloning extensions...')
    %cd -q $Extensions
    get_ipython().run_line_magic('clone', _tmp.name)
    os.unlink(_tmp.name)

# Embeddings + Upscalers
_asset_queue = []
for _url in [Embedding_1, Embedding_2]:
    if _url.strip(): _asset_queue.append((_url.strip(), str(Embeddings), None))
for _url in [Upscaler_1, Upscaler_2]:
    if _url.strip(): _asset_queue.append((_url.strip(), str(Upscalers), None))

if _asset_queue:
    if Parallel_Download:
        parallel_batch_download(_asset_queue, max_workers=Max_Workers)
    else:
        for _url, _dest, _fn in _asset_queue:
            %cd -q $_dest
            %download $_url

## ⚡ FLUX Models *(Optional)*
<span style="font-size:13px;">Works with <b>Forge</b>, <b>ComfyUI</b>, <b>SwarmUI</b>. Leave all empty if not needed.</span>

In [ ]:
# @title ⚡ **FLUX Model Downloader** {"display-mode":"form"}
# @markdown ### Select FLUX Variant
FLUX_Variant = 'None' # @param ["None", "FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]
FLUX_Unet   = 'https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-schnell-fp8.safetensors' # @param {type:"string"}
FLUX_Clip_L = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors' # @param {type:"string"}
FLUX_T5XXL  = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors' # @param {type:"string"}
FLUX_VAE    = 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors' # @param {type:"string"}
# @markdown ---
# @markdown ### ⚡ Speed Options
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:6, step:1}

from nenen88 import parallel_batch_download

if FLUX_Variant != 'None':
    _unet = FLUX_Unet.replace('schnell', 'dev') if 'dev' in FLUX_Variant.lower() and 'schnell' in FLUX_Unet else FLUX_Unet
    _flux_queue = [(u, d, f) for u, d, f in [
        (_unet,       str(UNET), None),
        (FLUX_Clip_L, str(CLIP), None),
        (FLUX_T5XXL,  str(CLIP), None),
        (FLUX_VAE,    str(VAE),  'flux_ae.safetensors'),
    ] if u.strip()]
    print(f'\n⚡ Downloading {FLUX_Variant} ({len(_flux_queue)} files)...')
    if Parallel_Download:
        parallel_batch_download(_flux_queue, max_workers=Max_Workers)
    else:
        for _url, _dest, _fn in _flux_queue:
            %cd -q $_dest
            if _fn:
                %download $_url $_fn
            else:
                %download $_url
    print('\n✅ FLUX ready. Enable FLUX support in your WebUI.')
else:
    print('  FLUX_Variant is "None" — skipping.')

## 🎛️ ControlNet *(Optional)*

In [ ]:
# @title 🎛️ **ControlNet Widget**
%run $Controlnet_Widget

# 🚀 Launch
<span style="font-size:14px;">
Common args per WebUI:<br>
• <b>A1111</b>: <code>--xformers</code><br>
• <b>Forge</b>: <code>--disable-xformers --opt-sdp-attention --cuda-stream</code><br>
• <b>ReForge</b>: <code>--xformers --cuda-stream</code><br>
• <b>ComfyUI</b>: <code>--dont-print-server --use-pytorch-cross-attention</code><br>
• Add <code>--N=ngrok_token</code> for NGROK or <code>--Z=zrok_token</code> for ZROK tunnel.
</span>

In [ ]:
# @title 🚀 **Launch WebUI** {"display-mode":"form"}
# @markdown ### Select WebUI arguments preset
WebUI_Arguments_Preset = 'Forge' # @param ["A1111", "Forge", "ReForge", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI", "Custom"]
# @markdown > Select the WebUI you installed to apply optimized presets automatically.
Extra_Args = '' # @param {type:"string", placeholder:"Any additional args (e.g. --medvram)"}
Ngrok_Token = '' # @param {type:"string", placeholder:"Your Ngrok token (optional)"}
Zrok_Token = '' # @param {type:"string", placeholder:"Your Zrok token (optional)"}
Skip_Widget = False # @param {type:"boolean"}
Skip_ComfyUI_Check = False # @param {type:"boolean"}
# @markdown > Check to skip ComfyUI node dependency checks (faster cold start).

presets = {
    'A1111': '--xformers',
    'Forge': '--disable-xformers --opt-sdp-attention --cuda-stream',
    'ReForge': '--xformers --cuda-stream',
    'Forge-Classic': '--xformers --cuda-stream --persistent-patches',
    'Forge-Neo': '--xformers --cuda-malloc --cuda-stream',
    'ComfyUI': '--dont-print-server --use-pytorch-cross-attention',
    'SwarmUI': '--launch_mode none',
    'Custom': ''
}

_args = presets.get(WebUI_Arguments_Preset, '')
if Extra_Args.strip(): _args += f" {Extra_Args.strip()}"
if Ngrok_Token.strip(): _args += f" --N={Ngrok_Token.strip()}"
if Zrok_Token.strip(): _args += f" --Z={Zrok_Token.strip()}"
if Skip_Widget: _args += ' --skip-widget'
if Skip_ComfyUI_Check: _args += ' --skip-comfyui-check'

%cd -q $WebUI
get_ipython().run_line_magic('run', f'segsmaker.py {_args}')